In [87]:
# Import necessary modules from your files
import os
import subprocess
import numpy as np

from tqdm.notebook import tqdm

from searchclient.agent_types.classic import * 

# Import all action classes (used for hardcoding solutions) and actions libraries
from searchclient.domains.hospital.actions import (
    NoOpAction, MoveAction, PushAction, PullAction, AnyAction, DEFAULT_MAPF_ACTION_LIBRARY, DEFAULT_HOSPITAL_ACTION_LIBRARY
)

# Import state, goal description and level classes for the MAvis hospital environment
from searchclient.domains.hospital.state import HospitalState
from searchclient.domains.hospital.goal_description import HospitalGoalDescription
from searchclient.domains.hospital.level import HospitalLevel

# Import the Graph-Search algorithm
from searchclient.search_algorithms.graph_search import graph_search

# Import the different search strategies for both uninformed and informed search
from searchclient.strategies.bfs import FrontierBFS
from searchclient.strategies.dfs import FrontierDFS
from searchclient.strategies.bestfirst import FrontierBestFirst, FrontierGreedy, FrontierAStar

# Import heuristic classes, to be used in informed search methods
from searchclient.domains.hospital.heuristics import (
    HospitalZeroHeuristic, HospitalGoalCountHeuristics, HospitalAdvancedHeuristics
)

from PIL import Image
from renderState import *

import importlib
import sys

# Remove the old cached module
if 'strategies.bestfirst' in sys.modules:
    del sys.modules['strategies.bestfirst']
if 'searchclient.search_algorithms.graph_search' in sys.modules:
    del sys.modules['searchclient.search_algorithms.graph_search']
if 'search_algorithms.graph_search' in sys.modules:
    del sys.modules['search_algorithms.graph_search']


# Reimport it fresh
from strategies.bestfirst import FrontierGreedy
from searchclient.search_algorithms.graph_search import graph_search


Function for Loading a Level

In [88]:
def load_level_file_from_path(path):
    with open(path, "r") as f:
        lines = f.readlines()
        lines = list(map(lambda line: line.strip(), lines))
        return lines

Level Path for the examples:

In [89]:
level_path = "levels/SimpleDebug.lvl"

Example Usage (```CTRL + A + CTRL + /``` to enable)

In [90]:
# # Example usage: load_level('path_to_level_file.lvl')
# level_lines = load_level_file_from_path(level_path)
# level = HospitalLevel.parse_level_lines(level_lines)

# # We can access the initial state of the level using the following code
# initial_state = HospitalState(level, level.initial_agent_positions, level.initial_box_positions)

# # We can access the goal description of the level using the following code
# goal_description = HospitalGoalDescription(level, level.box_goals + level.agent_goals)

# print('The initial state of the level is:')
# print(initial_state)

# print('\nThe goal description of the level is:')
# print(goal_description) # which tells us where the level objects (like boxes and agents) should be placed to satisfy the goal
# print('\nSo agent zero starts at {} and satisfies the goal at {}'.format(level.initial_agent_positions[0][0], goal_description.agent_goals[0][0]))

# # Render some state of the level (here the initial state)
# render_state(level_path=level_path, state=initial_state, output_path='rendering')

# # So the state looks like this in .txt format (what the computer uses):
# print(initial_state)

# # And the initial state looks like this in .png:
# # You can either show the image in the notebook or open it in a new window
# img = Image.open('rendering.png')
# img.show()

Function for Animated Visualization

In [91]:
def render_plan(level_path, plan, strategy_name, heuristic_name, num_generated, elapsed_time, sol_length):

    str_plan = convert_plan_to_string(plan) #convert the plan to a string

    # this just makes sure that the meta information is displayed correctly in the visualization
    if strategy_name == 'greedy' or strategy_name == 'astar':
        strategy_name_pygame = strategy_name + ' w. ' + heuristic_name
    else:
        strategy_name_pygame = strategy_name
    
    subprocess.run(["python3", 
                    "renderMAvis.py", 
                    "--level", level_path, 
                    "--plan", str_plan, 
                    "--search_strategy", strategy_name_pygame, 
                    "--num_generated", str(num_generated), 
                    "--time_elapsed", str(elapsed_time), 
                    "--sol_length", str(sol_length)])

In [92]:
# Here is an example of a plan. A plan is a list of list of actions, where an action is an instance from searchclient.domains.hospital.actions
# The first axis of the list corresponds to a timestep in the plan. 
# Each element of the innermost list corresponds to an action of a different agent. 
# Check the documentation of graph_seach.py for more information!
hardcoded_plan = [[MoveAction("E")], [NoOpAction()], [MoveAction("W")]]

# Hardcode your solution here!
#hardcoded_plan = None

render_plan(level_path, hardcoded_plan, "Hardcoded sol", None, len(hardcoded_plan), "0.0", len(hardcoded_plan))

pygame 2.6.1 (SDL 2.28.4, Python 3.12.3)
Hello from the pygame community. https://www.pygame.org/contribute.html


ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1342:(snd_func_refer) error evaluating name
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5727:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM default


Finished executing the plan


In [93]:
# Before running the search algorithm, we need to define the action set and the action library
# Use this library for pure pathfinding problems
action_library = DEFAULT_MAPF_ACTION_LIBRARY

# Use this library for sokoban-like problems (includes Push and Pull actions)
#action_library = DEFAULT_HOSPITAL_ACTION_LIBRARY

# Every agent will have the same action set
action_set = [action_library] * level.num_agents

# In order to run Graph-Search, we need to specify the initial state, action set, goal description, and frontier 

# If needed, we need to specify and fetch a heuristic function before initializing the frontier (informed search) 
# When adding new heuristics, remember to update the dictionary! 
# Use a string that matches the name of the heuristic function in the heuristics.py file
heuristic_name = "zero" 
heuristic = {
        'zero': HospitalZeroHeuristic,
        'goalcount': HospitalGoalCountHeuristics,
        'advanced': HospitalAdvancedHeuristics,
    }.get(heuristic_name, HospitalZeroHeuristic)() # make sure you understand what .get() does


# Finally, let's pick the search strategy and fetch the relevant frontier
strategy_name = "greedy" 
frontier = {
        'bfs': FrontierBFS,
        'dfs': FrontierDFS,
        'astar': lambda: FrontierAStar(heuristic),
        'greedy': lambda: FrontierGreedy(heuristic)
    }.get(strategy_name, FrontierBFS)() # make sure you understand what .get() does

In [94]:
# Now that we have defined the initial state, action set, goal description, and frontier, we can run the search algorithm
planning_success, plan, num_generated, elapsed_time = graph_search(initial_state, action_set, goal_description, frontier)

# The graph search function returns the following:
print('Planning successful:', planning_success)
print('Plan:', plan)
print('Solution length:', len(plan))
print('Number of states generated:', num_generated)
print('Elapsed time:', elapsed_time)


FRONTIER CONTENTS (Size: 1)
State                          g-value      h-value      f-value     
--------------------------------------------------------------------------------
++++
+0 +
+  +
++++            0            0            0           


FRONTIER CONTENTS (Size: 2)
State                          g-value      h-value      f-value     
--------------------------------------------------------------------------------
++++
+ 0+
+  +
++++            1            0            0           
++++
+  +
+0 +
++++            1            0            0           


FRONTIER CONTENTS (Size: 2)
State                          g-value      h-value      f-value     
--------------------------------------------------------------------------------
++++
+  +
+0 +
++++            1            0            0           
++++
+  +
+ 0+
++++            2            0            0           

#Expanded:        3, #Frontier:        1, #Generated:        4, Time: 0,003 s, Memory: 106,54 MB


Planning

In [95]:
# For easiness of use, let's redifine and reload everything  
level_path = "levels/SimpleDebug.lvl"
level_lines = load_level_file_from_path(level_path)
level = HospitalLevel.parse_level_lines(level_lines)

# We can access the initial state of the level using the following code
initial_state = HospitalState(level, level.initial_agent_positions, level.initial_box_positions)

# We can access the goal description of the level using the following code
goal_description = HospitalGoalDescription(level, level.box_goals + level.agent_goals)

print('The initial state of the level is:')
print(initial_state)

print('\nThe goal description of the level is:')
print(goal_description) # which tells us where the level objects (like boxes and agents) should be placed to satisfy the goal
print('\nSo agent zero starts at {} and satisfies the goal at {}'.format(level.initial_agent_positions[0][0], goal_description.agent_goals[0][0]))

The initial state of the level is:
++++
+0 +
+  +
++++

The goal description of the level is:
((2, 2), '0', True)

So agent zero starts at (1, 1) and satisfies the goal at (2, 2)


In [96]:
# Before running the search algorithm, we need to define the action set and the action library
# Use this library for pure pathfinding problems
action_library = DEFAULT_MAPF_ACTION_LIBRARY

# Use this library for sokoban-like problems (includes Push and Pull actions)
#action_library = DEFAULT_HOSPITAL_ACTION_LIBRARY

# Every agent will have the same action set
action_set = [action_library] * level.num_agents

# In order to run Graph-Search, we need to specify the initial state, action set, goal description, and frontier 

# If needed, we need to specify and fetch a heuristic function before initializing the frontier (informed search) 
# When adding new heuristics, remember to update the dictionary! 
# Use a string that matches the name of the heuristic function in the heuristics.py file
heuristic_name = "zero" 
heuristic = {
        'zero': HospitalZeroHeuristic,
        'goalcount': HospitalGoalCountHeuristics,
        'advanced': HospitalAdvancedHeuristics,
    }.get(heuristic_name, HospitalZeroHeuristic)() # make sure you understand what .get() does


# Finally, let's pick the search strategy and fetch the relevant frontier
strategy_name = "greedy" 
frontier = {
        'bfs': FrontierBFS,
        'dfs': FrontierDFS,
        'astar': lambda: FrontierAStar(heuristic),
        'greedy': lambda: FrontierGreedy(heuristic)
    }.get(strategy_name, FrontierBFS)() # make sure you understand what .get() does

In [97]:
n_trials = 10

plans, sol_lengths, generated, elapsed = [], [], [], []
for n in tqdm(range(n_trials)):
    planning_success, plan, num_generated, elapsed_time = graph_search(initial_state, action_set, goal_description, frontier)
    plans.append(plan)
    sol_lengths.append(len(plan))
    generated.append(int(num_generated))
    elapsed.append(elapsed_time)


# The graph search function returns the following:
print('Average solution legth:', np.mean(sol_lengths))
print('Solution length variance :', np.var(sol_lengths))
print('Average number of states generated:', np.mean(generated))
print('Number of states generated variance :', np.var(generated))

  0%|          | 0/10 [00:00<?, ?it/s]


FRONTIER CONTENTS (Size: 1)
State                          g-value      h-value      f-value     
--------------------------------------------------------------------------------
++++
+0 +
+  +
++++            0            0            0           


FRONTIER CONTENTS (Size: 2)
State                          g-value      h-value      f-value     
--------------------------------------------------------------------------------
++++
+  +
+0 +
++++            1            0            0           
++++
+ 0+
+  +
++++            1            0            0           


FRONTIER CONTENTS (Size: 2)
State                          g-value      h-value      f-value     
--------------------------------------------------------------------------------
++++
+ 0+
+  +
++++            1            0            0           
++++
+  +
+ 0+
++++            2            0            0           

#Expanded:        3, #Frontier:        1, #Generated:        4, Time: 0,001 s, Memory: 106,54 MB



FRONTIE

In [98]:
# In case you want to visualize the i-th trial

# Select trial to render
index_to_render = 0

# Render trial
render_plan(
    level_path,
    plans[index_to_render], 
    strategy_name, 
    heuristic_name, 
    generated[index_to_render],
    elapsed[index_to_render], 
    sol_lengths[index_to_render]
)


pygame 2.6.1 (SDL 2.28.4, Python 3.12.3)
Hello from the pygame community. https://www.pygame.org/contribute.html


ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1342:(snd_func_refer) error evaluating name
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5727:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM default


Finished executing the plan
